In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv("gurgaon_properties_post_feature_selection_v2.csv")

In [ ]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,3.25,3,3,2,Old Property,2250.0,0,0,0,Low,Low Floor
1,flat,sector 67,2.70,4,4,3+,Moderately Old,2127.0,1,0,1,Medium,Mid Floor
2,house,sector 109,6.75,4,4,3+,New Property,6228.0,1,0,0,Medium,Low Floor
3,flat,sohna road,0.38,2,2,2,Relatively New,750.0,0,0,0,Low,Mid Floor
4,flat,sector 74,1.78,2,2,3+,Relatively New,1582.0,0,0,0,Medium,High Floor


In [ ]:
df['furnishing_type'].value_counts()

,count
furnishing_type,
0,2352
1,1018
2,187


In [ ]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0:'unfurnished',1:'semifurnished',2:'furnished'})

In [ ]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,3.25,3,3,2,Old Property,2250.0,0,0,unfurnished,Low,Low Floor
1,flat,sector 67,2.70,4,4,3+,Moderately Old,2127.0,1,0,semifurnished,Medium,Mid Floor
2,house,sector 109,6.75,4,4,3+,New Property,6228.0,1,0,unfurnished,Medium,Low Floor
3,flat,sohna road,0.38,2,2,2,Relatively New,750.0,0,0,unfurnished,Low,Mid Floor
4,flat,sector 74,1.78,2,2,3+,Relatively New,1582.0,0,0,unfurnished,Medium,High Floor


In [ ]:
df['sector'].value_counts()

,count
sector,
sohna road,146
sector 85,108
sector 102,107
sector 92,100
sector 69,93
...,...
sector 88b,3
sector 73,3
sector 27,2


In [ ]:
a=list(df[(df['sector']=='sector 27')|(df['sector']=='sector 17a')|(df['sector']=='sector 37')].index)

In [ ]:
df.drop(a,inplace=True)

In [ ]:
X=df.drop(columns=['price'])
y=df['price']

In [ ]:
# Applying the log1p transformation to the target column
y_transformed = np.log1p(y)

## **Ordinal Encoding**

In [ ]:
columns_to_encode = ['property_type','sector','balcony','agePossession','furnishing_type','luxury_category','floor_category']

In [ ]:
from sklearn.preprocessing import OrdinalEncoder , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(transformers=[
    ('cat',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=113),columns_to_encode),
    ('num',StandardScaler(),['bedRoom','bathroom','built_up_area','servant room','store room'])
], remainder='passthrough')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold , cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [ ]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [ ]:
kfold=KFold(n_splits=10,shuffle=True,random_state=42)
scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')

In [ ]:
scores.mean(),scores.std()

(np.float64(0.7369935545371538), np.float64(0.0345977260121625))

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=113),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred=pipeline.predict(X_test)

In [ ]:
y_pred=np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9399977717936052

In [ ]:
def scorer(model_name,model):

  output=[]
  output.append(model_name)

  pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',model)
  ])

  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')
  output.append(scores.mean())

  X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

  pipeline.fit(X_train,y_train)
  y_pred=pipeline.predict(X_test)
  y_pred=np.expm1(y_pred)
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [30]:
from sklearn.svm import SVR
from sklearn.linear_model import Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output=[]
for model_name,model in model_dict.items():
  model_output.append(scorer(model_name,model))

In [ ]:
model_output

[['linear_reg', np.float64(0.7369935545371538), 0.9399977717936052],
 ['svr', np.float64(0.7611940616441191), 0.8876742243795843],
 ['ridge', np.float64(0.7369945739731467), 0.9397343670042089],
 ['LASSO', np.float64(0.053007424880547216), 1.6091854047111647],
 ['decision tree', np.float64(0.7894044856679486), 0.7117402212249433],
 ['random forest', np.float64(0.8835502630702562), 0.5668079829489578],
 ['extra trees', np.float64(0.8694676331773792), 0.6017170830666151],
 ['gradient boosting', np.float64(0.8752301399945033), 0.6185941167819001],
 ['adaboost', np.float64(0.7573268445348496), 0.8608505842004773],
 ['mlp', np.float64(0.8100695554898827), 0.7368786526924792],
 ['xgboost', np.float64(0.8924473587765442), 0.5627599940957231]]

In [ ]:
model_df=pd.DataFrame(model_output,columns=['Name','R2','MAE'])

In [ ]:
model_df.sort_values('MAE')

,Name,R2,MAE
10,xgboost,0.892447,0.562760
5,random forest,0.883550,0.566808
6,extra trees,0.869468,0.601717
7,gradient boosting,0.875230,0.618594
4,decision tree,0.789404,0.711740
9,mlp,0.810070,0.736879
8,adaboost,0.757327,0.860851
1,svr,0.761194,0.887674
2,ridge,0.736995,0.939734
0,linear_reg,0.736994,0.939998


## **OneHotEncoding**

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num',StandardScaler(),['bedRoom','bathroom','built_up_area','servant room','store room']),
    ('cat',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=113),columns_to_encode),
    ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','furnishing_type'])
],remainder='passthrough')

In [ ]:
X.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,3,3,2,Old Property,2250.0,0,0,unfurnished,Low,Low Floor
1,flat,sector 67,4,4,3+,Moderately Old,2127.0,1,0,semifurnished,Medium,Mid Floor
2,house,sector 109,4,4,3+,New Property,6228.0,1,0,unfurnished,Medium,Low Floor
3,flat,sohna road,2,2,2,Relatively New,750.0,0,0,unfurnished,Low,Mid Floor
4,flat,sector 74,2,2,3+,Relatively New,1582.0,0,0,unfurnished,Medium,High Floor


In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.855940592983039)

In [ ]:
scores.std()

np.float64(0.025698051059524572)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=113),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sector', 'agePossession',
                                                   'furnishing_type'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
y_pred = np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6491096973221276

In [ ]:
def scorer(model_name,model):

  output=[]
  output.append(model_name)

  pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',model)
  ])

  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')
  output.append(scores.mean())

  X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

  pipeline.fit(X_train,y_train)
  y_pred=pipeline.predict(X_test)
  y_pred=np.expm1(y_pred)
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [ ]:
model_output=[]
for model_name,model in model_dict.items():
  model_output.append(scorer(model_name,model))

In [ ]:
model_output

[['linear_reg', np.float64(0.855940592983039), 0.6491096973221276],
 ['svr', np.float64(0.7655865559718151), 0.8823716797057765],
 ['ridge', np.float64(0.8562622252865332), 0.6530204243121058],
 ['LASSO', np.float64(0.05300742488054726), 1.6091854047111644],
 ['decision tree', np.float64(0.8067020873413842), 0.6971293808887918],
 ['random forest', np.float64(0.8912442644977844), 0.539177325212952],
 ['extra trees', np.float64(0.8957171152070895), 0.4930628026969561],
 ['gradient boosting', np.float64(0.8758338002569992), 0.6106741754607504],
 ['adaboost', np.float64(0.7614334404800729), 0.876965497310866],
 ['mlp', np.float64(0.8723970285349825), 0.5644758379942905],
 ['xgboost', np.float64(0.8948498151160162), 0.5632841596086988]]

In [ ]:
model_df=pd.DataFrame(model_output,columns=['Name','R2','MAE'])

In [ ]:
model_df.sort_values(['MAE'])

,Name,R2,MAE
6,extra trees,0.895717,0.493063
5,random forest,0.891244,0.539177
10,xgboost,0.894850,0.563284
9,mlp,0.872397,0.564476
7,gradient boosting,0.875834,0.610674
0,linear_reg,0.855941,0.649110
2,ridge,0.856262,0.653020
4,decision tree,0.806702,0.697129
8,adaboost,0.761433,0.876965
1,svr,0.765587,0.882372


## **OneHotEncoding With PCA**

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num',StandardScaler(),['bedRoom','bathroom','built_up_area','servant room','store room']),
    ('cat',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=113),columns_to_encode),
    ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),['sector','agePossession','furnishing_type'])
],remainder='passthrough')

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.05617260655867802)

In [ ]:
scores.std()

np.float64(0.021989759160866915)

In [ ]:
def scorer(model_name,model):

  output=[]
  output.append(model_name)

  pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',model)
  ])

  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')
  output.append(scores.mean())

  X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

  pipeline.fit(X_train,y_train)
  y_pred=pipeline.predict(X_test)
  y_pred=np.expm1(y_pred)
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_df=pd.DataFrame(model_output,columns=['Name','R2','MAE'])

In [ ]:
model_df.sort_values('MAE')

,Name,R2,MAE
6,extra trees,0.896893,0.497176
10,xgboost,0.898001,0.539928
5,random forest,0.892536,0.541466
9,mlp,0.867986,0.562635
7,gradient boosting,0.875985,0.611259
0,linear_reg,0.855932,0.649163
2,ridge,0.856422,0.652690
4,decision tree,0.800435,0.657122
8,adaboost,0.759424,0.861027
1,svr,0.765587,0.882372


## **Target Encoder**

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 4.1 MB/s eta 0:00:00


In [ ]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(handle_unknown='value'), ['sector'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean(),scores.std()

(np.float64(0.8287318573516487), np.float64(0.027781503400571162))

In [ ]:
def scorer(model_name,model):

  output=[]
  output.append(model_name)

  pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',model)
  ])

  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')
  output.append(scores.mean())

  X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

  pipeline.fit(X_train,y_train)
  y_pred=pipeline.predict(X_test)
  y_pred=np.expm1(y_pred)
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_df=pd.DataFrame(model_output,columns=['Name','R2','MAE'])

In [ ]:
model_df.sort_values('MAE')

,Name,R2,MAE
6,extra trees,0.903104,0.464945
5,random forest,0.903232,0.474844
10,xgboost,0.904173,0.486978
7,gradient boosting,0.889030,0.546000
9,mlp,0.847718,0.607761
4,decision tree,0.826183,0.635590
0,linear_reg,0.828732,0.695996
2,ridge,0.828757,0.696399
8,adaboost,0.818604,0.714973
1,svr,0.779386,0.861414


## **Hyperparameter Tuning**

In [19]:
from sklearn.model_selection import GridSearchCV

In [27]:
param_grid = {
    'regressor__n_estimators': [50,100,200,300],
    'regressor__learning_rate': [0.05, 0.1,0.15,0.2],
    'regressor__max_depth': [3,4,5,6,8],
    'regressor__gamma':[0,0.1,0.2,0.3],
    'regressor__min_child_weight': [1,3,5,7]
}

In [28]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(handle_unknown='value'), ['sector'])
    ],
    remainder='passthrough'
)

In [31]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_jobs=-1,tree_method='hist',random_state=42))
])

In [32]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [33]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

In [34]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 1280 candidates, totalling 12800 fits


GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['bedRoom',
                                                                          'bathroom',
                                                                          'built_up_area',
                                                                          'servant '
                                                                          'room',
                                                                          'store '
                                                                          'room']),
                                                                        ('cat',
                                                                         OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                        unknown_value=-1),
                                                                         ['property_type',
                                                                          's...
                                                     monotone_constraints=None,
                                                     multi_strategy=None,
                                                     n_estimators=None,
                                                     n_jobs=-1,
                                                     num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'regressor__gamma': [0, 0.1, 0.2, 0.3],
                         'regressor__learning_rate': [0.05, 0.1, 0.15, 0.2],
                         'regressor__max_depth': [3, 4, 5, 6, 8],
                         'regressor__min_child_weight': [1, 3, 5, 7],
                         'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='r2', verbose=4)

In [35]:
final_pipe = search.best_estimator_

In [36]:
search.best_params_

{'regressor__gamma': 0,
 'regressor__learning_rate': 0.1,
 'regressor__max_depth': 5,
 'regressor__min_child_weight': 1,
 'regressor__n_estimators': 300}

In [37]:
search.best_score_

np.float64(0.907103744877783)

In [38]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [39]:
final_pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_ca...
                              feature_types=None, feature_weights=None, gamma=0,
                              grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None, min_child_weight=1,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=300, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [40]:
y_pred=final_pipe.predict(X_test)

In [41]:
y_pred=np.expm1(y_pred)

In [42]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.493187041220618

In [54]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(handle_unknown='value'), ['sector'])
    ],
    remainder='passthrough'
)

In [536]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(learning_rate=0.1,max_depth=5,n_estimators=600,tree_method='hist',subsample=0.71))
])

In [537]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [538]:
scores.mean()

np.float64(0.9069774837342841)

In [539]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [540]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_ca...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=600, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [541]:
y_pred=pipeline.predict(X_test)

In [542]:
y_pred=np.expm1(y_pred)

In [543]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.47233882894495877

In [545]:
pipeline.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_ca...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=600, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [547]:
import pickle

with open('pipeline.pkl','wb') as file:
  pickle.dump(pipeline,file)

In [548]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [549]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,3,3,2,Old Property,2250.0,0,0,unfurnished,Low,Low Floor
1,flat,sector 67,4,4,3+,Moderately Old,2127.0,1,0,semifurnished,Medium,Mid Floor
2,house,sector 109,4,4,3+,New Property,6228.0,1,0,unfurnished,Medium,Low Floor
3,flat,sohna road,2,2,2,Relatively New,750.0,0,0,unfurnished,Low,Mid Floor
4,flat,sector 74,2,2,3+,Relatively New,1582.0,0,0,unfurnished,Medium,High Floor
...,...,...,...,...,...,...,...,...,...,...,...,...
3552,flat,sector 90,2,2,3,Relatively New,1225.0,0,0,semifurnished,Low,High Floor
3553,flat,sector 70,3,3,2,Relatively New,2167.0,0,0,unfurnished,Medium,Mid Floor
3554,house,sector 82,4,4,3+,Relatively New,3240.0,1,1,unfurnished,Low,Mid Floor
3555,flat,sohna road,1,1,1,New Property,443.0,0,0,furnished,Low,Mid Floor


In [566]:
data = [['house', 'sector 49', 3, 3, '3+', 'New Property', 1750, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 49,3,3,3+,New Property,1750,0,0,unfurnished,Low,Low Floor


In [567]:
np.expm1(pipeline.predict(one_df))

array([2.527026], dtype=float32)